In [4]:
# Cell 1: Imports and setup
import os
import sys
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from config import *
from utils import set_seed, generate_persian_reasoning
from data_loader import OHLCVDataset, create_dataloader, load_image
from smc_engine import smc_analysis, smc_feature_vector
from transformer import OHLCVTransformer, ViT, TimeframeFusion
from model import MarketPredictor
from backtester import Backtester
from trading_engine import LocalTradingEngine
from ui import TradingApp

set_seed(42)
print(f"Using device: {DEVICE}")

Using device: cpu


In [5]:
# Cell 2: Generate synthetic OHLCV data (for demonstration)
def generate_synthetic_ohlcv(n_candles=2000, trend='mixed'):
    np.random.seed(0)
    close = 50000 + np.cumsum(np.random.randn(n_candles) * 200)
    high = close + np.abs(np.random.randn(n_candles) * 100)
    low = close - np.abs(np.random.randn(n_candles) * 100)
    open = close - np.random.randn(n_candles) * 50
    volume = np.random.rand(n_candles) * 100
    df = pd.DataFrame({'open': open, 'high': high, 'low': low, 'close': close, 'volume': volume})
    return df

df = generate_synthetic_ohlcv(2000)
df.to_csv('synthetic_btc.csv', index=False)
print("Synthetic data saved.")
df.head()

Synthetic data saved.


,open,high,low,close,volume
0,50250.683657,50506.102575,50291.475553,50352.810469,71.680219
1,50478.814970,50604.038927,50248.471913,50432.841911,1.214030
2,50622.856006,50633.203014,50601.480409,50628.589508,59.667020
3,51083.639333,51172.605596,50963.123385,51076.768148,87.089198
4,51382.003399,51458.360907,51276.446559,51450.279746,69.840843


In [6]:
# Cell 3: Load dataset and create dataloader
train_loader = create_dataloader('synthetic_btc.csv', batch_size=32, train=True)
test_loader = create_dataloader('synthetic_btc.csv', batch_size=32, train=False)
sample_batch = next(iter(train_loader))
print(f"Batch X shape: {sample_batch[0].shape}, y shape: {sample_batch[1].shape}")

Batch X shape: torch.Size([32, 64, 4]), y shape: torch.Size([32])


In [ ]:
# Cell 4: Initialize model, optimizer, and train
model = MarketPredictor(use_image=False).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = torch.nn.CrossEntropyLoss()

model.train()
losses = []
for epoch in range(EPOCHS):
    epoch_loss = 0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        # Prepare multi-timeframe dict (simulate by resampling original data)
        # For simplicity, we use the same 1m data for all timeframes
        ohlcv_dict = {tf: x for tf in TIMEFRAMES}
        # Generate SMC features from last window (batch)
        smc_feats = []
        for i in range(x.size(0)):
            # convert sequence to df
            seq = x[i].cpu().numpy()
            df_snippet = pd.DataFrame(seq, columns=['open','high','low','close'])
            smc_res = smc_analysis(df_snippet)
            feat = smc_feature_vector(smc_res, window_size=SEQ_LEN)
            smc_feats.append(feat)
        smc_feats = torch.tensor(smc_feats, dtype=torch.float32).to(DEVICE)
        
        logits, confidence = model(ohlcv_dict=ohlcv_dict, smc_features=smc_feats)
        loss = criterion(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    losses.append(epoch_loss / len(train_loader))
    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {epoch_loss/len(train_loader):.4f}")

C:\Users\Shayan\AppData\Local\Temp\ipykernel_14884\2163741048.py:24: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  smc_feats = torch.tensor(smc_feats, dtype=torch.float32).to(DEVICE)


In [ ]:
# Cell 5: Evaluate model
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        ohlcv_dict = {tf: x for tf in TIMEFRAMES}
        smc_feats = []
        for i in range(x.size(0)):
            seq = x[i].cpu().numpy()
            df_snippet = pd.DataFrame(seq, columns=['open','high','low','close'])
            smc_res = smc_analysis(df_snippet)
            smc_feats.append(smc_feature_vector(smc_res, window_size=SEQ_LEN))
        smc_feats = torch.tensor(smc_feats, dtype=torch.float32).to(DEVICE)
        logits, _ = model(ohlcv_dict=ohlcv_dict, smc_features=smc_feats)
        preds = torch.argmax(logits, dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)
print(f"Test Accuracy: {correct/total:.3f}")

In [ ]:
# Cell 6: Persian inference on a sample
model.eval()
x, y = next(iter(test_loader))
x, y = x.to(DEVICE), y.to(DEVICE)
ohlcv_dict = {tf: x for tf in TIMEFRAMES}
smc_feats = []
for i in range(x.size(0)):
    seq = x[i].cpu().numpy()
    df_snippet = pd.DataFrame(seq, columns=['open','high','low','close'])
    smc_res = smc_analysis(df_snippet)
    smc_feats.append(smc_feature_vector(smc_res, window_size=SEQ_LEN))
smc_feats = torch.tensor(smc_feats, dtype=torch.float32).to(DEVICE)

logits, confidence = model(ohlcv_dict=ohlcv_dict, smc_features=smc_feats)
probs = torch.softmax(logits, dim=-1)
pred_class = torch.argmax(probs, dim=1)
classes = ['LONG','SHORT','NO_TRADE']

# For the first sample
idx = 0
seq_df = pd.DataFrame(x[idx].cpu().numpy(), columns=['open','high','low','close'])
smc_info = smc_analysis(seq_df)
trend = "bullish" if pred_class[idx]==0 else "bearish"
decision = classes[pred_class[idx].item()]
conf = confidence[idx].item()
reason = generate_persian_reasoning(decision, conf, smc_info, trend)
print(reason)

In [ ]:
# Cell 7: Run backtest on full dataset
backtester = Backtester(df, model, smc_func=smc_analysis)
metrics = backtester.run()
print(json.dumps(metrics, indent=2))

In [ ]:
# Cell 8: Launch PyQt5 UI (optional – run separately if desired)
# Uncomment to launch GUI (will block notebook)
# app = QtWidgets.QApplication(sys.argv)
# window = TradingApp()
# window.show()
# sys.exit(app.exec_())

In [ ]:
# Cell 9: Attention visualization (example using a dummy transformer layer)
def plot_attention_weights(model, sample, layer=-1):
    # Attach hook to access attention weights
    attn_weights = []
    def hook(module, input, output):
        # output is attn_output, attn_weights from multihead attention
        if isinstance(output, tuple) and len(output) > 1:
            attn_weights.append(output[1].detach().cpu())
    handles = []
    for name, module in model.named_modules():
        if 'attention' in name.lower() or 'attn' in name.lower():
            handles.append(module.register_forward_hook(hook))
    with torch.no_grad():
        model.eval()
        # run a forward pass through the base transformer (simplified)
        # This is illustrative; actual hook depends on your model structure.
    for h in handles:
        h.remove()
    if attn_weights:
        plt.imshow(attn_weights[0][0,0].numpy(), cmap='viridis')
        plt.colorbar()
        plt.title("Self-Attention Map")
        plt.show()
    else:
        print("No attention weights captured.")
# Example: plot_attention_weights(model.tf_encoders['1m'].transformer.layers[0].self_attn, sample_batch[0])
print("Attention visualisation available.")